# 5.0 · Análisis Exploratorio de Datos (EDA)

**Tras preprocesamiento.** Se mueve aquí el EDA completo (antes en 2.0/2.1). Usa base **con_outliers** con outliers **ocultos en gráficas**. Separa **calidad (15%)** y **hallazgos de negocio (20%)** según guía Sesion 5.

In [ ]:
from pathlib import Path
import sys

def _carpeta_refactor() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'lib' / 'config.py').exists(): return p
        if (p / 'notebooks' / 'refactor' / 'lib' / 'config.py').exists(): return p / 'notebooks' / 'refactor'
    raise RuntimeError('No encontrado')
RAIZ=_carpeta_refactor(); sys.path.insert(0,str(RAIZ))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from lib import calidad, carga, categorias, competencias, config, figuras, periodo
%matplotlib inline
print('ok')


In [ ]:
df = carga.leer(config.CON_OUTLIERS)
periodo.validar(df)
df['period_time']=periodo.crear_period_time(df)
df['duracion']=periodo.duracion_observada(df)
df['nivel_ordinal']=categorias.nivel_ordinal(df)
df['nivel_no_reportado']=categorias.nivel_no_reportado(df)
df['isco_mayor']=categorias.isco_mayor(df)
df=competencias.agregar_banderas(df)
competencias.validar(df)
lim_dur=calidad.limites_iqr(df['duracion'])
print('Base:', len(df))


## 1. Calidad de datos (15%)

Se reporta: integridad, duplicados, nulos, outliers (conservados pero ocultados).

In [ ]:
print(calidad.resumen_duplicados(df))
display(calidad.resumen_vacios(df))
print('Outliers duración:', int(calidad.marcar_outliers(df['duracion']).sum()))


## 2. Hallazgos de negocio (20%)

In [ ]:
display(df['isco_mayor'].value_counts().sort_index().to_frame('filas'))
display(df['university_level'].value_counts().to_frame('filas'))
fig=figuras.histograma(calidad.ocultar(df['duracion']), 'Duración (oculta para legibilidad)', 'Trimestres', bins=50)
plt.show()
display(competencias.resumen_banderas(df))


## 3. Conclusiones

- **Falso positivo corregido:** usar `saber_skills`, no listas vacías.
- **Banderas diferenciadas:** `estado_competencias` y `estado_conocimientos` responden preguntas distintas.
- **Outliers:** 35.477 reales; conservados en base con_outliers, ocultados en gráficas.
- **`Period_Time`:** solo descriptivo, nunca predictor.
- **Listas:** no entran a modelo; información resumida en contadores.